<a href="https://colab.research.google.com/github/DVORA-AZARKOVICH/Narrative-Similarity/blob/main/NarrativeSimilarity_story_emb_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Narrative Similarity - Inference on Test Set
**SemEval 2026 - Task 4**

This notebook performs the **Inference** stage using the fine-tuned model.
The workflow includes:
1.  Setting up the environment and dependencies.
2.  Loading the Test Data.
3.  Loading the Base Model (`story-emb`) combined with the trained **LoRA Adapter**.
4.  Generating embeddings and calculating similarity scores.
5.  Creating the final submission file.

In [ ]:
!pip install -q transformers==4.57.3 peft==0.18.0 bitsandbytes==0.49.0 accelerate==1.12.0 spacy
!python -m spacy download en_core_web_lg

import torch
import os
print(f"CUDA Available: {torch.cuda.is_available()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
CUDA Available: True


### 1. Load Data & Define Paths
Here we mount Google Drive to access the saved model and the test dataset.
We load the test file (`test_track_a.jsonl`), which contains the triplets: **Anchor**, **Text A**, and **Text B**.

In [ ]:
from google.colab import drive
import pandas as pd
import json

drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Narrative Similarity Data/'
MODEL_PATH = BASE_PATH + "fine_tuned_story_emb_final" # הנתיב בו שמרת את המודל
TEST_FILE = BASE_PATH + 'SemEval2026-Task_4-test-v1/test_track_a.jsonl'

df_test = pd.read_json(TEST_FILE, lines=True)
print("\n--- Test Data Loaded ---")
print(f"Shape: {df_test.shape}")
print(df_test.head(2))

Mounted at /content/drive

--- Test Data Loaded ---
Shape: (400, 3)
                                         anchor_text  \
0  Having been a victim of bullying, life at his ...   
1  Florida, 1969. 8-year-old Tommy Wheeler is inc...   

                                              text_a  \
0  Alberto is an employee who is the Italian aver...   
1  When orphaned Jimmy Mason is taken in by his A...   

                                              text_b  
0  A lab mix-up accidentally swaps a vaccine with...  
1  Set in the mid-1960s, the story centers on ten...  


### 2. Load Base Model & Fine-Tuned Adapter
This step loads the model components:
1.  **Quantization Config (BNB):** The base model (`uhhlt/story-emb`) is loaded in **4-bit precision** to save memory.
2.  **Base Model:** The pre-trained foundation model.
3.  **PeftModel:** We load the **trained adapter weights** (from the fine-tuning stage) and attach them to the base model.
4.  `model.eval()`: Sets the model to evaluation mode (disabling dropout).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

base_model_name = "uhhlt/story-emb"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"Loading Fine-Tuned Adapter from: {MODEL_PATH}")
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model.eval()

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

Loading Fine-Tuned Adapter from: /content/drive/MyDrive/Narrative Similarity Data/fine_tuned_story_emb_final
Model loaded successfully!


### 3. Inference & Similarity Calculation
We iterate through the test set to predict which text is narratively closer to the anchor:
1.  **Generate Embeddings:** The model processes the Anchor, Text A, and Text B. We extract the representation of the **Last Token** to represent the entire story.
2.  **Normalization:** Embeddings are normalized (L2) to ensure valid cosine similarity comparisons.
3.  **Scoring:** We calculate the dot product (similarity) between:
    * Anchor & Text A
    * Anchor & Text B
4.  **Decision:** If `Score(A) > Score(B)`, the prediction is `True`.

In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm
tqdm.pandas()

def get_last_token_embeddings(outputs, attention_mask):
    hidden_states = outputs.hidden_states[-1]
    last_token_indices = attention_mask.sum(1) - 1
    batch_size = hidden_states.shape[0]
    return hidden_states[torch.arange(batch_size), last_token_indices]

def predict_row(row):
    anchor = "Retrieve stories with a similar narrative to the given story: " + str(row['anchor_text'])
    txt_a = str(row['text_a'])
    txt_b = str(row['text_b'])

    with torch.no_grad():
        inputs = tokenizer(
            [anchor, txt_a, txt_b],
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=1024
        ).to(model.device)

        outputs = model(**inputs, output_hidden_states=True)
        embeddings = get_last_token_embeddings(outputs, inputs.attention_mask)
        embeddings = F.normalize(embeddings, p=2, dim=1)

        # embeddings[0] = anchor
        # embeddings[1] = text_a
        # embeddings[2] = text_b

        score_a = torch.dot(embeddings[0], embeddings[1]).item()
        score_b = torch.dot(embeddings[0], embeddings[2]).item()


print("Running prediction on Test set...")
df_test['prediction'] = df_test.progress_apply(predict_row, axis=1)
print("Prediction finished.")

Running prediction on Test set...


  0%|          | 0/400 [00:00<?, ?it/s]

Prediction finished.


### 4. Format Submission & Export
Finally, we prepare the data for submission:
1.  Rename the prediction column to `text_a_is_closer` (as required by the competition format).
2.  Save the results to a `JSONL` file.
3.  Compress the file into a `ZIP` archive and trigger a download.

In [ ]:
import zipfile
from google.colab import files

submission_df = df_test[['prediction']].copy()
submission_df = submission_df.rename(columns={'prediction': 'text_a_is_closer'})

if 'id' in df_test.columns:
    submission_df['id'] = df_test['id']

jsonl_filename = 'track_a.jsonl'
submission_df.to_json(jsonl_filename, orient='records', lines=True)

zip_filename = 'submission_model+test.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(jsonl_filename)

print(f"Created {zip_filename}")
files.download(zip_filename)

Created submission_test.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>